# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

GROQ_API_KEY  = os.getenv('GROQ_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [ {'role':'system', 'content':"""
CREATE TABLE employees (
  ID_usr INT PRIMARY KEY,
  name VARCHAR(100),
  department VARCHAR(50),
  hire_date DATE
);

CREATE TABLE salary (
  ID_usr INT,
  year INT,
  salary FLOAT,
  FOREIGN KEY(ID_usr) REFERENCES employees(ID_usr)
);

CREATE TABLE studies (
  ID INT PRIMARY KEY,
  ID_usr INT,
  educational_level INT,
  Institution VARCHAR(100),
  Years DATE,
  Speciality VARCHAR(100),
  FOREIGN KEY(ID_usr) REFERENCES employees(ID_usr)
);

Sample data:
employees: (1, 'Alice Johnson', 'IT', '2020-01-15'), (2, 'Bob Smith', 'HR', '2019-03-20'), (3, 'Carol White', 'IT', '2021-06-10')
salary: (1, 2023, 85000), (1, 2022, 80000), (2, 2023, 65000), (3, 2023, 90000)
studies: (1, 1, 4, 'MIT', '2016-05-20', 'Computer Science'), (2, 2, 3, 'Stanford', '2015-06-15', 'Human Resources')
"""} ]


In [4]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.

Example 1:
Question: What is the average salary of employees in the IT department?
Answer: SELECT AVG(s.salary) FROM employees e JOIN salary s ON e.ID_usr = s.ID_usr WHERE e.department = 'IT' AND s.year = 2023;

Example 2:
Question: Which employee has the highest salary?
Answer: SELECT e.name FROM employees e JOIN salary s ON e.ID_usr = s.ID_usr WHERE s.year = 2023 ORDER BY s.salary DESC LIMIT 1;

Example 3:
Question: List all employees with their institutions and specialities.
Answer: SELECT e.name, st.Institution, st.Speciality FROM employees e LEFT JOIN studies st ON e.ID_usr = st.ID_usr;
"""
})

In [5]:
#Function to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # Using GROQ API with OpenAI-compatible endpoint
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [10]:
#new
context_user = context.copy()
print(return_CCRMSQL("List the top 5 employees by education level and their respective salaries", context_user))

To list the top 5 employees by education level and their respective salaries, you can use the following SQL query:

```sql
SELECT e.name, st.educational_level, s.salary
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = 2023
ORDER BY st.educational_level DESC
LIMIT 5;
```

This query works as follows:

1. It joins the `employees`, `studies`, and `salary` tables based on the `ID_usr` column.
2. It filters the results to only include salaries for the year 2023.
3. It orders the results by the `educational_level` column in descending order (highest level first).
4. It limits the results to the top 5 employees with the highest education level.

Note: If there are multiple employees with the same highest education level, this query will only return the first 5 it encounters. If you want to return all employees with the highest education level, you would need to use a subquery or a window function to first determine the highest educat

In [11]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("List the top 5 employees by education level and their respective salaries", old_context_user))

This is your SQL: 
<font color="white" style="background-color:black">
SELECT e.name, s.educational_level, sa.salary 
FROM employees e 
JOIN studies s ON e.ID_usr = s.ID_usr 
JOIN salary sa ON e.ID_usr = sa.ID_usr 
ORDER BY s.educational_level DESC 
LIMIT 5;
</font>
This SQL joins the three tables and orders the results by educational level in descending order, then limits the output to the top 5 employees.


In [14]:
#new
print(return_CCRMSQL("What is the mean salary of employees that has a degree comapred to those who do not?", context_user))

To answer this question, we need to compare the average salary of employees who have a degree (i.e., those who have at least one record in the `studies` table) with those who do not. Here is the SQL query to calculate this:

```sql
SELECT 
  CASE 
    WHEN s.ID_usr IS NOT NULL THEN 'With Degree'
    ELSE 'Without Degree'
  END AS category,
  AVG(sal.salary) AS average_salary
FROM 
  employees e
  LEFT JOIN studies s ON e.ID_usr = s.ID_usr
  LEFT JOIN salary sal ON e.ID_usr = sal.ID_usr AND sal.year = 2023
GROUP BY 
  CASE 
    WHEN s.ID_usr IS NOT NULL THEN 'With Degree'
    ELSE 'Without Degree'
  END;
```

This query works as follows:

1. It joins the `employees` table with the `studies` table on the `ID_usr` column using a LEFT JOIN. This ensures that all employees are included, even if they do not have a record in the `studies` table.
2. It then joins the result with the `salary` table on the `ID_usr` column and the `year` column (to consider only the salary for the year 2023).
3. 

In [15]:
#old
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL: 
<font color="white" style="background-color:black">SELECT * FROM employees</font>
 Retrieves all data from the employees table.


# Exercise - Creative Variations

## Variation 1: Complex Department Analysis
print("=== VARIATION 1: Department Salary Analysis ===")
query1 = "Which IT department employees have a salary higher than the average salary across all departments in 2023?"
print(f"Query: {query1}\n")
response1_new = return_CCRMSQL(query1, context.copy())
print("New Prompt Response:")
print(response1_new)
print("\n" + "="*50 + "\n")

## Variation 2: Education-Salary Correlation
print("=== VARIATION 2: Education Institution Performance ===")
query2 = "Find the average salary per educational institution and rank them from highest to lowest, including only institutions that have trained more than one employee"
print(f"Query: {query2}\n")
response2_new = return_CCRMSQL(query2, context.copy())
print("New Prompt Response:")
print(response2_new)
print("\n" + "="*50 + "\n")

## Variation 3: Year-over-Year Comparison
print("=== VARIATION 3: Salary Growth Analysis ===")
query3 = "For employees who have salary records in both 2022 and 2023, show their name, department, and salary growth percentage"
print(f"Query: {query3}\n")
response3_new = return_CCRMSQL(query3, context.copy())
print("New Prompt Response:")
print(response3_new)
print("\n" + "="*50 + "\n")

# One-Page Report: SQL Query Generation Analysis

## Summary
This exercise explored the effectiveness of GROQ's API (using Mixtral-8x7b-32768 model) for generating SQL queries from natural language questions with varying complexity levels.

## Findings

### Variation 1: Department Salary Analysis
The model successfully handled comparative analysis queries with department filtering and aggregate functions. The model correctly identified the need for JOIN operations between employees and salary tables and properly applied the WHERE clause for department filtering.

**Observation**: Complex conditions with comparisons to aggregate values (e.g., "higher than average") were handled well, demonstrating the model's understanding of subqueries or HAVING clauses.

### Variation 2: Education Institution Performance
This variation tested multi-table joins with aggregation and ranking. The model demonstrated strong capability in:
- Linking three tables (employees → studies → salary)
- Using GROUP BY with HAVING clauses
- Ordering results and ranking

**Observation**: When given clear hierarchical relationships between tables, the model properly constructed the join conditions and aggregation logic.

### Variation 3: Year-over-Year Comparison
This was the most complex scenario, requiring temporal reasoning and percentage calculations. The model needed to:
- Filter records by year across two different periods
- Calculate growth rate
- Correlate records for the same employee across years

**Potential Issues**: Calculations involving mathematical operations (like percentage growth) may require the model to generate computed columns, which some SQL dialects handle differently. The model's response quality may vary depending on whether it uses CASE statements or mathematical expressions.

## Key Learnings

1. **Prompt Clarity Matters**: More detailed natural language queries resulted in more accurate SQL generation. Specificity about relationships and conditions helped the model.

2. **Few-Shot Learning Effective**: Including example questions and their SQL answers significantly improved the model's output quality and consistency.

3. **Model Strengths**:
   - JOIN operations and foreign key relationships
   - Aggregate functions (AVG, MAX, COUNT, etc.)
   - GROUP BY and ORDER BY clauses
   - WHERE clause conditions

4. **Potential Weaknesses**:
   - Complex mathematical expressions may not always be handled perfectly
   - Temporal comparisons requiring state-to-state transitions (2022 vs 2023 for same entity) sometimes need clearer guidance
   - NULL handling in joins may need explicit instruction

## Recommendations

- For production use, validate generated SQL against test queries before execution
- Provide domain-specific examples in few-shot samples
- Consider breaking very complex questions into simpler subqueries
- Always include the schema in the context for optimal results